In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile")


## Main Components of any RAG based application 

1. Document Loaders
2. Text Splitters
3. Vector Stores
4. LLMs
5. RetrievalQA Chain


Different types of Document Loaders :

`TextLoader, PyPDFLoader, WebBaseLoader, CSVLoader`

Langchain provides a `Document` class to represent the documents that are loaded by the Document Loaders. The `Document` class has two main attributes: 
`page_content` and `metadata`. The `page_content` attribute contains the actual content of the document, while the `metadata` attribute contains any additional information about the document, such as its source or author.

```python
Document(
    page_content='This is the content of the document.',
    metadata={'source': 'filename.pdf'}
)
```

# 1. TextLoader

> For loading .txt files

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("poem.txt", encoding="utf8")

docs = loader.load()

print(docs)

print(f"Type of Documents: {type(docs)}")
print(f"Type of Document in Documents: {type(docs[0])}")
print(f"Number of Documents: {len(docs)}\n")

print(f"Actual Data:\n{docs[0].page_content}\n")
print(f"Meta Data:\n{docs[0].metadata}\n")


[Document(metadata={'source': 'poem.txt'}, page_content='Under the sun, the pitch stands still,\nA battle of patience, power, and will.\nThe bowler runs with fire in eyes,\nThe bat swings hope toward open skies.\n\nCheers erupt with every score,\nA six that sails, a crowd that roars.\nFrom dusty grounds to grand stadium light,\nCricket lives in hearts, day and night. 🏏')]
Type of Documents: <class 'list'>
Type of Document in Documents: <class 'langchain_core.documents.base.Document'>
Number of Documents: 1

Actual Data:
Under the sun, the pitch stands still,
A battle of patience, power, and will.
The bowler runs with fire in eyes,
The bat swings hope toward open skies.

Cheers erupt with every score,
A six that sails, a crowd that roars.
From dusty grounds to grand stadium light,
Cricket lives in hearts, day and night. 🏏

Meta Data:
{'source': 'poem.txt'}



# 2. PyPDFLoader

> processes by pages and returns a list of `Document` objects, each containing the content of a single page. The `metadata` attribute of each `Document` object contains the page number and the source of the document.

```python
from langchain_community.document_loaders import PyPDFLoader

loader2 = PyPDFLoader("sample.pdf")

docs2 = loader2.load()

print(len(docs2))

print(docs[0].page_content[:500])
print(docs[0].metadata)
```

## Choosing the right PDF Loader

|Use Case| Recommended Loader|
|---|---|
|Simple, clean pdfs| PyPDFLoader|
|Scanned / image PDFs| UnstructuredPDFLoader or AmazonTextractPDFLoader|
|PDFs with tables/columns| PDFPlumberLoader|
|Need layout and image data| PyMuPDFLoader|
| Want best structure extraction| UnstructuredPDFLoader|

---

# 3. DirectoryLoader

> It is a document loader that allows you to load all the documents from a directory. It takes a directory path as input and loads all the documents in that directory using the appropriate document loader based on the file type. For example, if the directory contains a mix of .txt and .pdf files, it will use `TextLoader` for .txt files and `PyPDFLoader` for .pdf files.

```python
loader = DirectoryLoader(
    path = 'data/', 
    glob='**/*.pdf'),    -> for loading all pdf files in data folder and its subfolders
    loader_cls=PyPDFLoader
)
```

---

## Load vs Lazy Load
- `load()`: This method loads all the documents into memory at once. It is suitable for small to medium-sized datasets where memory constraints are not an issue. It allows for faster access to the documents since they are already loaded in memory.

Best when - no. of documents is small , and you want everything loaded upfront.

- `lazy_load()`: This method loads documents on demand, meaning it only loads a document when it is accessed. It can help reduce memory usage and improve performance when working with large collections of documents.

Returns: A **generator** of `Document` objects instead of a list, which allows you to iterate over the documents without loading them all into memory at once.

Best when - when you want streaming

***

## 4. WebBaseLoader  

![loader](https://img.shields.io/badge/loader-WebBaseLoader-blue)  

WebBaseLoader fetches a web page and returns it as one or more `Document` objects (page content + `metadata`). It’s a great first step for scraping static pages and quick crawls.

> **How it works:** sends an HTTP request (via `requests`) and parses HTML with `BeautifulSoup` to extract readable text and basic metadata (source URL, title).

```python
from langchain_community.document_loaders import WebBaseLoader

url = "https://example.com/article"
loader = WebBaseLoader(url)
docs = loader.load()
print(docs[0].metadata.get('source'))
print(docs[0].page_content[:400])
```

✅ <strong>Pro tip:</strong> After loading, run a text splitter (e.g., `RecursiveCharacterTextSplitter`) before creating embeddings to keep chunks semantic and small.

⚠️Limitations:WebBaseLoader does not execute JavaScript — it won’t render dynamic content. For JS-heavy sites, use <strong>SeleniumURLLoader</strong> or an API/official feed where available.</div>

When to use:
- Quick ingestion of static blog posts, docs, and simple HTML pages.
- Prototyping or small-scale scraping where rendering isn't required.

When not to use:
- Pages that rely on client-side rendering (SPAs) or lazy-loaded content.
- Large-scale, production-grade crawling — use a robust scraper or pipeline.


In [7]:
from langchain_community.document_loaders import WebBaseLoader

loader3 = WebBaseLoader("https://www.w3schools.com/howto/howto_website_static.asp")

docs3 = loader3.load()

print(docs3)

[Document(metadata={'source': 'https://www.w3schools.com/howto/howto_website_static.asp', 'title': 'How To Make a Static Website', 'language': 'en-US'}, page_content="\n\n\n\nHow To Make a Static Website\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n        Tutorials\n        \n\n\n\n        References\n        \n\n\n\n        Exercises\n        \n\n\n\n        Certificates\n        \n\n\n\n\n      Menu\n      \n\n\n\n\n\n          Search field\n        \n\n\n\n\n\n\n\n\n\n\n\n\n\n\n×\n\n\n\n\n\n\n          See More\n        \n\n\n\n\xa0\n\n\n\n\n\n\n\n\n\n\n\nSign In\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n★\n+1\n\n\n\n\n\n\n\n\n\n        Get Certified\n      \n\n\n\n\n\n\n        Upgrade\n      \n\n\n\n\n        Teachers\n      \n\n\n\n\n        Spaces\n      \n\n\n\n\n        Bootcamps\n      \n\n\n\n\n\n\n        Get Certified\n      \n\n\n\n\n\n\n        Upgrade\n      \n\n\n\n\n        Teachers\n      \n\n\n\n\n        Spa